In [1]:
import json
import re
import time
import pandas as pd
import numpy as np
import os

from tqdm.notebook import tqdm

from selenium import webdriver
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

In [2]:
key_lst = [
    'periodMinuteLimits',
    'attendance',
    'referee',
    'weatherCode',
    'startTime',
    'score',
    'home',
    'away',
    'events'  
]

def match_Info_Extraction(driver, url):
    
    driver.get(url)
    WebDriverWait(driver, 20).until(
        lambda d: "matchCentreData" in d.page_source
    )
    html = driver.page_source
    m = re.search(
        r"matchCentreData\s*:\s*(\{.*?\})\s*,\s*matchCentreEventTypeJson",
        html,
        flags=re.S
    )

    start = html.find("matchCentreData")
    start = html.find("{", start)
    depth = 0
    for i in range(start, len(html)):
        c = html[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break
                
    raw_json = html[start:end]
    data = json.loads(raw_json)
    
    try:
        sub_dict = {k: data[k] for k in key_lst}
        match = url.split("/")[-1]
        if not os.path.exists("Data"):
            os.mkdir("Data")
        with open(f"Data/{match}.json", "w", encoding="utf-8") as f:
            json.dump(
                sub_dict,
                f,
                ensure_ascii=False,
                indent=4
            )
    except Exception as e:
        print(f"Failed: {url}")
        print(type(e).__name__, e)
        return None
    
def match_Info(driver, match_url_lst):
    for url in tqdm(match_url_lst):
        time.sleep(np.random.uniform(2, 5))
        d = match_Info_Extraction(driver, url)

In [17]:
with open("all_urls_bundesliga_2.json", "r", encoding="utf-8") as f:
    all_urls_set = json.load(f)

In [19]:
%%time

options = Options()
driver = webdriver.Chrome(options=options)

match_Info(driver, all_urls_set["Germany_Bundesliga_2"][2497:])
driver.quit()

  0%|          | 0/869 [00:00<?, ?it/s]

Failed: https://www.whoscored.com/matches/1743705/live/germany-2-bundesliga-2023-2024-hertha-berlin-st-pauli
KeyError 'periodMinuteLimits'
CPU times: total: 4min 45s
Wall time: 1h 25min 36s
